# Filesystem, Code Execution, and Sandbox Security

Build and evaluate a brokered, one-use execution boundary for an agent that analyzes uploaded archives. You will attack archive admission, runtime capabilities, resource policy, replay handling, and degraded-mode behavior while preserving valid work.

## Safety contract

This notebook never executes generated code, extracts an archive, starts a process, reads host files for the simulated workload, or performs network I/O. Programs are typed effect fixtures interpreted by an independent deterministic runtime. This makes the security decisions testable without pretending that Python objects are production isolation.

In [ ]:
from dataclasses import asdict, replace
import asyncio
import importlib.util
import json
from pathlib import Path
import sys

course_path = Path('curriculum/roadmap/intermediate/05-filesystem-code-execution-and-sandbox-security')
lab_path = next(path for path in (Path.cwd() / 'lab.py', Path.cwd() / course_path / 'lab.py') if path.exists())
spec = importlib.util.spec_from_file_location('course05_sandbox_lab', lab_path)
assert spec and spec.loader
LAB = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = LAB
spec.loader.exec_module(LAB)
print(f'Loaded deterministic lab from {lab_path.name}')

## The boundary to preserve

![A model supplies only archive and program IDs. Trusted context, archive admission, a one-use brokered grant, a disposable runtime, four independent policy planes, bounded output, redacted evidence, and verified destruction form the security boundary.](architecture.svg)

The model proposes two identifiers. Authenticated application state supplies the workload and tenant. Trusted registries supply immutable input and program digests. Policy and image configuration are never model arguments.

In [ ]:
scenario = LAB.build_scenario()
policy_view = {
    'version': scenario.policy.version,
    'image_digest': scenario.policy.image_digest,
    'network_enabled': scenario.policy.network_enabled,
    'max_cpu_ticks': scenario.policy.max_cpu_ticks,
    'max_memory_bytes': scenario.policy.max_memory_bytes,
    'max_pids': scenario.policy.max_pids,
    'max_written_bytes': scenario.policy.max_written_bytes,
    'max_output_bytes': scenario.policy.max_output_bytes,
}
policy_view

## Part 1 — Break naive path checks

A prefix and substring check looks plausible until encoding and link semantics enter the system. Compare that baseline with canonical admission.

In [ ]:
encoded_escape = 'input/%2e%2e/%2e%2e/etc/passwd'
link_archive = LAB.archive_fixture(
    'archive:north:link', 'north',
    (LAB.ArchiveMember('input/link', LAB.MemberKind.SYMLINK, target='../../host'),),
)
baseline = {
    'encoded_escape_accepted': LAB.unsafe_path_baseline(encoded_escape),
    'symlink_archive_accepted': LAB.unsafe_archive_baseline(link_archive),
}
assert all(baseline.values())
baseline

In [ ]:
path_attempts = [
    '../host', '%2e%2e/host', '%252e%252e/host', '..\\host',
    '/etc/passwd', 'C:\\Windows\\system.ini',
    'input/%2525252e%2525252e/secret', 'input/data/orders.csv',
]
path_results = {path: LAB.canonical_relative_path(path) for path in path_attempts}
assert path_results['input/data/orders.csv'][0] == 'input/data/orders.csv'
assert all(result[0] is None for path, result in path_results.items() if path != 'input/data/orders.csv')
path_results

In [ ]:
safe_archive = scenario.archives.archives['archive:north:safe']
safe_admission, safe_reason = scenario.inspector.inspect(safe_archive)
link_admission, link_reason = scenario.inspector.inspect(link_archive)
bomb = LAB.archive_fixture(
    'archive:north:bomb', 'north',
    (LAB.ArchiveMember('data.csv', LAB.MemberKind.FILE, 100_001, 1),),
)
bomb_admission, bomb_reason = scenario.inspector.inspect(bomb)
assert safe_admission and safe_reason == 'archive-admitted'
assert link_admission is None and link_reason == 'archive-links-denied'
assert bomb_admission is None and bomb_reason == 'archive-compression-ratio'
{'safe_paths': sorted(safe_admission.paths), 'safe_manifest': safe_admission.manifest_digest[:16], 'link': link_reason, 'bomb': bomb_reason}

## Part 2 — Preserve a useful job

A security control that blocks every request is not useful. Run the valid North tenant path and inspect the bounded report and lifecycle evidence.

In [ ]:
scenario = LAB.build_scenario()
valid = scenario.application.analyze_archive(
    attestation_id='attest:north',
    archive_id='archive:north:safe',
    program_id='program:safe',
    request_id='notebook:valid',
    now=LAB.NOW,
)
assert valid.status is LAB.DecisionStatus.ALLOW
assert json.loads(valid.report)['status'] == 'complete'
assert scenario.runtime.host_read_count == 0
assert scenario.runtime.host_write_count == 0
assert scenario.runtime.network_dispatch_count == 0
{'status': valid.status.value, 'reason': valid.reason, 'report': json.loads(valid.report)}

In [ ]:
execution_receipt = valid.receipts[-1]
receipt = asdict(execution_receipt)
forbidden_evidence_fields = {'file_content', 'source_code', 'credential', 'host_path'}
assert not forbidden_evidence_fields.intersection(receipt)
assert execution_receipt.sandbox_created and execution_receipt.sandbox_destroyed
assert all(record.destroyed for record in scenario.runtime.records.values())
receipt

## Part 3 — Keep authorization and runtime policy independent

Cross-tenant input is denied before a sandbox exists. Filesystem, network, syscall, and resource effects are denied after an authorized grant enters the disposable runtime.

In [ ]:
scenario = LAB.build_scenario()
cross_tenant = scenario.application.analyze_archive(
    attestation_id='attest:north', archive_id='archive:south:safe',
    program_id='program:safe', request_id='notebook:tenant', now=LAB.NOW,
)
assert cross_tenant.status is LAB.DecisionStatus.DENY
assert cross_tenant.reason == 'archive-tenant'
assert not cross_tenant.receipts[-1].sandbox_created
{'status': cross_tenant.status.value, 'reason': cross_tenant.reason, 'sandbox_created': False}

In [ ]:
filesystem_results = {}
for program_id in ('program:encoded-read', 'program:absolute-read'):
    scenario = LAB.build_scenario()
    response = scenario.application.analyze_archive(
        attestation_id='attest:north', archive_id='archive:north:safe',
        program_id=program_id, request_id=f'notebook:{program_id}', now=LAB.NOW,
    )
    filesystem_results[program_id] = (response.status.value, response.reason, response.receipts[-1].sandbox_destroyed)
assert all(status == 'deny' and destroyed for status, _, destroyed in filesystem_results.values())
filesystem_results

In [ ]:
capability_results = {}
for program_id in ('program:network', 'program:syscall'):
    scenario = LAB.build_scenario()
    response = scenario.application.analyze_archive(
        attestation_id='attest:north', archive_id='archive:north:safe',
        program_id=program_id, request_id=f'notebook:{program_id}', now=LAB.NOW,
    )
    capability_results[program_id] = response.reason
assert capability_results == {'program:network': 'sandbox-network-denied', 'program:syscall': 'sandbox-syscall-denied'}
capability_results

In [ ]:
resource_programs = ('program:pids', 'program:cpu', 'program:memory', 'program:disk', 'program:output')
resource_results = {}
for program_id in resource_programs:
    scenario = LAB.build_scenario()
    response = scenario.application.analyze_archive(
        attestation_id='attest:north', archive_id='archive:north:safe',
        program_id=program_id, request_id=f'notebook:{program_id}', now=LAB.NOW,
    )
    resource_results[program_id] = response.reason
assert all(reason.startswith('resource-') for reason in resource_results.values())
resource_results

## Part 4 — Bind, consume, and invalidate authority

The grant is short-lived, integrity-protected, current-state checked, and one-use. It binds the authenticated sender key as well as the exact archive, manifest, program, policy, and image.

In [ ]:
scenario = LAB.build_scenario()
issued = scenario.broker.issue(
    'attest:north',
    LAB.ExecutionRequest('notebook:grant', 'archive:north:safe', 'program:safe'),
    now=LAB.NOW,
)
grant = issued.grant
assert grant and scenario.broker.verify_integrity(grant)
{name: getattr(grant, name) for name in ('workload_id', 'tenant', 'archive_id', 'program_id', 'policy_version', 'image_digest', 'expires_at')}

In [ ]:
first = scenario.runtime.execute('attest:north', grant.grant_id, request_id='notebook:first', now=LAB.NOW)
second = scenario.runtime.execute('attest:north', grant.grant_id, request_id='notebook:replay', now=LAB.NOW)
assert first.status is LAB.DecisionStatus.ALLOW
assert second.status is LAB.DecisionStatus.DENY and second.reason == 'execution-grant-replayed'
{'first': (first.status.value, first.reason), 'replay': (second.status.value, second.reason)}

In [ ]:
scenario = LAB.build_scenario()
issued = scenario.broker.issue(
    'attest:north', LAB.ExecutionRequest('notebook:failure', 'archive:north:safe', 'program:safe'), now=LAB.NOW,
)
scenario.runtime.available = False
failure = scenario.runtime.execute('attest:north', issued.grant.grant_id, request_id='notebook:failure:execute', now=LAB.NOW)
assert failure.status is LAB.DecisionStatus.ERROR
assert failure.reason == 'sandbox-runtime-unavailable'
assert scenario.application.fallback_execution_count == 0
{'status': failure.status.value, 'reason': failure.reason, 'host_fallbacks': scenario.application.fallback_execution_count}

## Part 5 — Evaluate the complete contract

Run the fixed 24-case suite. Severe security outcomes are reported independently from valid-job utility, dependency failures, cleanup, and trace completeness.

In [ ]:
report, observations = LAB.evaluate_controls()
assert (report.cases, report.valid_cases, report.attack_cases, report.failure_cases) == (24, 4, 18, 2)
assert report.valid_job_completion_rate == 1.0
assert report.escape_success_rate == 0.0
assert report.forbidden_read_success_rate == 0.0
assert report.resource_limit_enforcement_rate == 1.0
assert report.network_attempt_success_rate == 0.0
assert report.replay_acceptance_rate == 0.0
assert report.failure_error_preservation_rate == 1.0
assert report.cleanup_completeness_rate == 1.0
assert report.trace_completeness_rate == 1.0
asdict(report)

In [ ]:
case_table = [
    {
        'case': item.case_id, 'kind': item.kind.value, 'status': item.status,
        'created': item.sandbox_created, 'destroyed': item.sandbox_destroyed,
    }
    for item in observations
]
assert len(case_table) == 24
case_table

## Part 6 — Verify the OpenAI Agents SDK boundary

The model-facing strict tool accepts only `archive_id` and `program_id`. Identity, tenant, paths, network mode, resource limits, runtime image, grants, and sandbox lifecycle remain in trusted application context. The demo invokes the real tool wrapper without calling a model or API.

In [ ]:
sdk_path = lab_path.with_name('sdk_adapter.py')
sdk_spec = importlib.util.spec_from_file_location('course05_sandbox_sdk', sdk_path)
assert sdk_spec and sdk_spec.loader
SDK = importlib.util.module_from_spec(sdk_spec)
sys.modules[sdk_spec.name] = SDK
sdk_spec.loader.exec_module(SDK)
sdk_proof = await SDK.sdk_boundary_demo_async()
assert sdk_proof['strict_schema']
assert sdk_proof['trusted_fields_absent_from_schema']
assert sdk_proof['internal_handles_absent_from_results']
assert sdk_proof['safe_terminal_state'] == 'allow'
assert sdk_proof['network_terminal_state'] == 'deny'
assert sdk_proof['cross_tenant_terminal_state'] == 'deny'
sdk_proof

## Production transfer

Replace the typed interpreter with an assessed hardened container, gVisor runtime, microVM, WASI runtime, or hosted sandbox. Replace in-memory grants with transactional one-use state and managed key protection. Preserve the invariant: trusted derivation, immutable admission, exact capabilities, independent quotas, deny-by-default egress, bounded output, redacted evidence, explicit kill/destruction, and orphan reconciliation.

A provider SDK is an integration surface, not proof of containment. Pin versions and test timeout-versus-kill behavior, credential injection, egress defaults, image provenance, tenant isolation, cleanup under failure, and audit completeness.

## Checkpoint and next experiment

Why do a temporary directory, a safe-system prompt, and a client timeout still fail the governing invariant? Identify the independent component that constrains filesystem, network, syscall, resource, and lifecycle effects. Then add one exercise from the README—transactional redemption, wall-time semantics, a narrow package-fetch capability, or CSV output validation—and preserve all 24 baseline results.

In [ ]:
completion = {
    'unsafe_baseline_demonstrated': report.unsafe_baseline_escape_rate == 1.0,
    'valid_utility_preserved': report.valid_job_completion_rate == 1.0,
    'escapes_blocked': report.escape_success_rate == 0.0,
    'resources_enforced': report.resource_limit_enforcement_rate == 1.0,
    'cleanup_complete': report.cleanup_completeness_rate == 1.0,
    'sdk_boundary_verified': sdk_proof['trusted_fields_absent_from_schema'],
}
assert all(completion.values())
completion